# Draft reason-code map for compliance review (spec 13.2)

Maps every L4-pool feature to a reason group (RC01, RC02, ...) with a neutral customer-facing phrase, using the AECB and FTS source dictionaries. Writes `config/reason_code_map_DRAFT.csv` only; a reviewer saves the approved version as `config/reason_code_map.csv` (human-maintained, never written here).

In [0]:
%pip install optbinning "numpy<2" "pyarrow==14.0.1" "statsmodels>=0.14.4" "scipy>=1.12,<1.14" --quiet
dbutils.library.restartPython()

In [0]:
import sys, os, json, re, time
from datetime import datetime
sys.path.insert(0, os.getcwd())          # notebook lives in src/
import common
PROJECT_ROOT = common.find_project_root(os.getcwd())

import numpy as np
import pandas as pd

cfg = common.load_config(os.path.join(PROJECT_ROOT, "config", "config.yaml"))
SEED = cfg["random_seed"]
np.random.seed(SEED)

ARTIFACTS  = common.artifacts_path(cfg)
REPORTS    = os.path.join(ARTIFACTS, "reports")
RUN_SUMS   = os.path.join(REPORTS, "run_summaries")
FEAT_LISTS = os.path.join(ARTIFACTS, "feature_lists")
CONFIG_DIR = os.path.join(PROJECT_ROOT, "config")
os.makedirs(RUN_SUMS, exist_ok=True)

cat, sch = cfg["databricks"]["catalog"], cfg["databricks"]["schema"]
FLOG_TBL = f"{cat}.{sch}.acard_feature_log"

AECB_DICT = os.path.join(CONFIG_DIR, "aecb_features dict.csv")
FTS_DICT  = os.path.join(CONFIG_DIR, "fts_features_dict.csv")
DRAFT_CSV = os.path.join(CONFIG_DIR, "reason_code_map_DRAFT.csv")
FINAL_CSV = os.path.join(CONFIG_DIR, "reason_code_map.csv")      # human-maintained; never written here
FINAL_EXISTED_BEFORE = os.path.exists(FINAL_CSV)
FINAL_MTIME_BEFORE = os.path.getmtime(FINAL_CSV) if FINAL_EXISTED_BEFORE else None

GIT_COMMIT = common.git_commit(PROJECT_ROOT)
CONFIG_HASH = common.config_hash(cfg)
_t0 = time.time()
print(f"Project root: {PROJECT_ROOT}")
print(f"Git commit  : {GIT_COMMIT[:12]} | Config hash: {CONFIG_HASH[:12]}")
print(f"reason_code_map.csv exists before run: {FINAL_EXISTED_BEFORE}")

In [0]:
# encoding utf-8-sig strips the BOM on the first header
aecb_raw = pd.read_csv(AECB_DICT, encoding="utf-8-sig", dtype=str)
fts_raw  = pd.read_csv(FTS_DICT, encoding="utf-8-sig", dtype=str)
for name, d in (("AECB", aecb_raw), ("FTS", fts_raw)):
    print(f"{name} dictionary: {len(d)} rows | columns: {list(d.columns)}")
    display(d.head(5))

# Field name and description columns used for matching
NAME_COL, DESC_COL = "Feature Name", "Description"
assert {NAME_COL, DESC_COL} <= set(aecb_raw.columns) and {NAME_COL, DESC_COL} <= set(fts_raw.columns)
for name, d in (("AECB", aecb_raw), ("FTS", fts_raw)):
    k = d[NAME_COL].str.strip().str.lower()
    print(f"{name}: blank names {int(k.isna().sum())}, duplicate names {int(k.dropna().duplicated().sum())}, "
          f"blank descriptions {int(d[DESC_COL].isna().sum())}")

In [0]:
with open(os.path.join(FEAT_LISTS, "l4_pool.json")) as f:
    POOL = [r["feature"] for r in json.load(f)]
SPECIAL_RE = re.compile(r"_is_special_")
SPECIAL_IND = [f for f in POOL if SPECIAL_RE.search(f)]
POOL_FEATS = [f for f in POOL if not SPECIAL_RE.search(f)]
print(f"L4 pool: {len(POOL)} entries | special-value indicators excluded: {len(SPECIAL_IND)} | features: {len(POOL_FEATS)}")

flog = spark.table(FLOG_TBL).toPandas()
need = ["feature", "iv_dev", "final_A", "trend"]
missing_cols = [c for c in need if c not in flog.columns]
assert not missing_cols, f"feature log lacks columns {missing_cols}"
assert flog["feature"].is_unique, "feature log has duplicate feature rows"
assert set(POOL_FEATS) <= set(flog["feature"]), "pool features missing from the feature log"
flog_p = flog.set_index("feature").loc[POOL_FEATS, need[1:]]

fdict = pd.read_csv(os.path.join(CONFIG_DIR, "feature_dict.csv")).set_index("feature")

def _lookup(d):
    d = d.dropna(subset=[NAME_COL]).copy()
    d["_key"] = d[NAME_COL].str.strip().str.lower()
    return d.drop_duplicates("_key").set_index("_key")
AECB_LK, FTS_LK = _lookup(aecb_raw), _lookup(fts_raw)

def match(feat):
    """aecb_: strip the prefix; fts_: compare the full name (dictionary keeps the prefix). Case-insensitive."""
    src = fdict.loc[feat, "source"] if feat in fdict.index else None
    src = str(src) if pd.notna(src) and str(src).strip() else ""
    fl = feat.lower()
    if fl.startswith("aecb_"):
        k = fl[len("aecb_"):]
        return src or "AECB", k, (AECB_LK.loc[k, DESC_COL] if k in AECB_LK.index else None)
    if fl.startswith("fts_"):
        return src or "FTS", fl, (FTS_LK.loc[fl, DESC_COL] if fl in FTS_LK.index else None)
    return src or "OTHER", fl, None          # casa_, app_, AGE: no dictionary here

rows = []
for f in POOL_FEATS:
    src, key, desc = match(f)
    rows.append({"feature": f, "source": src, "match_key": key, "matched": desc is not None,
                 "dict_description": desc})
M = pd.DataFrame(rows).join(flog_p, on="feature")
print(f"Matched {int(M.matched.sum())}/{len(M)}")
pd.set_option("display.max_colwidth", 120)
display(M)

In [0]:
import shutil, tempfile

# ── Reason groups in the order given by the request; RC codes are assigned only to groups that are used. ──
# Each rule: (group, text_en, regex on dict_description [DICTIONARY], regex on feature name [NAME_ONLY]).
# Rules are evaluated in this order and the first match wins, so e.g. "Revolving OS/income" is debt level (not
# income) and "salary / credit" is salary credits (not cash flow). Time window / aggregation do not change a group.
GROUP_RULES = [
    ("enquiries",              "Number of recent credit applications",
     r"inquir|enquir",                                         r"inquir|enquir"),
    ("delinquency",            "Missed or late payments on accounts",
     r"overdue|\bdpd\b|delinq|arrear|write.?off",               r"overdue|dpd|delinq"),
    ("utilisation",            "Use of available credit card limits",
     r"utili[sz]|\butil\b|util[>%]",                            r"util"),
    ("debt level",             "Level of existing debt and repayments",
     r"unsecured|obligation|revolving|outstanding|\bos\b|exposure|limit", r"exposure|obligation|payment|limit|debt"),
    ("credit history length",  "Length of credit history",
     r"\bmob\b|first.ever contract|history|months on book",    r"_mob|file_age|history"),
    ("balances",               "Account balance levels and changes",
     r"balance",                                               r"balance"),
    ("salary credits",         "Salary payments into the account",
     r"salary",                                                r"salary|sal_"),
    ("cash flow",              "Pattern of money paid into and out of the account",
     r"credit|debit|cash.?flow|\bnet\b|fee|penalty|transaction", r"credit|debit|cashflow|net|fee|txn"),
    ("income",                 "Reported income",
     r"income",                                                r"income"),
    ("employment",             "Employment details",
     r"employ|employer|occupation|job",                         r"employ|occupation|job"),
    ("age",                    "Applicant age",
     r"\bage\b|date of birth",                                  r"(^|_)age($|_)|dob|birth"),
]
RULES = pd.DataFrame(GROUP_RULES, columns=["reason_group_name", "text_en", "desc_regex", "name_regex"])

def assign(row):
    if row.matched:
        text, col, by = str(row.dict_description).lower(), "desc_regex", "DICTIONARY"
    else:
        text, col, by = row.feature.lower(), "name_regex", "NAME_ONLY"
    for r in RULES.itertuples():
        if re.search(getattr(r, col), text):
            return r.reason_group_name, by
    return None, by

M[["reason_group_name", "suggested_by"]] = M.apply(lambda r: pd.Series(assign(r)), axis=1)
unassigned = M.loc[M.reason_group_name.isna(), "feature"].tolist()
assert not unassigned, f"No rule matched: {unassigned}. Extend GROUP_RULES or log an open question."

used = [g for g in RULES.reason_group_name if g in set(M.reason_group_name)]
RC_OF = {g: f"RC{i:02d}" for i, g in enumerate(used, start=1)}
TEXT_OF = dict(zip(RULES.reason_group_name, RULES.text_en))
M["reason_group"] = M.reason_group_name.map(RC_OF)
M["text_en"] = M.reason_group_name.map(TEXT_OF)
M["text_ar"] = ""

OUT_COLS = ["feature", "reason_group", "text_en", "text_ar", "source", "dict_description",
            "iv_dev", "final_A", "trend", "suggested_by"]
DRAFT = (M.assign(_o=M.reason_group.str[2:].astype(int))
          .sort_values(["_o", "iv_dev"], ascending=[True, False])[OUT_COLS].reset_index(drop=True))

# Write to local temp, then copy into config/ (DRAFT only; reason_code_map.csv is never written here)
assert os.path.basename(DRAFT_CSV) == "reason_code_map_DRAFT.csv"
_tmp = os.path.join(tempfile.mkdtemp(), "reason_code_map_DRAFT.csv")
DRAFT.to_csv(_tmp, index=False, encoding="utf-8")
shutil.copyfile(_tmp, DRAFT_CSV)
print(f"Wrote {DRAFT_CSV} ({len(DRAFT)} rows)\n")

# ── Summary ──
GROUP_SUMMARY = (DRAFT.assign(group=DRAFT.reason_group.map({v: k for k, v in RC_OF.items()}))
                 .groupby(["reason_group", "group", "text_en"], sort=True)
                 .agg(n_features=("feature", "size"), n_in_model_A=("final_A", lambda s: int((s == "Y").sum())),
                      features=("feature", ", ".join)).reset_index())
UNMATCHED = DRAFT.loc[DRAFT.suggested_by == "NAME_ONLY", ["feature", "reason_group", "text_en"]]
SINGLETONS = GROUP_SUMMARY.loc[GROUP_SUMMARY.n_features == 1, ["reason_group", "group", "features"]]
print(f"Number of groups: {len(GROUP_SUMMARY)} | groups with no pool feature: "
      f"{[g for g in RULES.reason_group_name if g not in RC_OF]}")
display(GROUP_SUMMARY)
print(f"Unmatched features (NAME_ONLY): {len(UNMATCHED)}")
display(UNMATCHED)
print(f"Single-feature groups: {len(SINGLETONS)}")
display(SINGLETONS)
display(DRAFT)

In [0]:
# ── Acceptance (draft reason-code map, spec 13.2) ──
print("=" * 60); print("ACCEPTANCE CRITERIA (s05 — draft reason-code map)"); print("=" * 60)
back = pd.read_csv(DRAFT_CSV, dtype={"text_ar": str}, keep_default_na=False, na_values=[""])
back["text_ar"] = back["text_ar"].fillna("")
exp = DRAFT.copy()
exp["text_ar"] = exp["text_ar"].fillna("")
rc_nums = sorted(int(c[2:]) for c in DRAFT.reason_group.unique())
matched_of = M.set_index("feature").matched
final_now = os.path.exists(FINAL_CSV)
final_mtime_now = os.path.getmtime(FINAL_CSV) if final_now else None

checks = [
    ("Columns are exactly as specified", list(back.columns) == OUT_COLS),
    (f"One row per L4-pool feature ({len(POOL_FEATS)})",
     len(DRAFT) == len(POOL_FEATS) and DRAFT.feature.is_unique and set(DRAFT.feature) == set(POOL_FEATS)),
    ("No special-value indicators", not DRAFT.feature.str.contains("_is_special_").any()),
    ("Every row has reason_group and text_en",
     DRAFT.reason_group.notna().all() and DRAFT.text_en.fillna("").str.strip().ne("").all()),
    (f"RC codes consecutive RC01..RC{len(rc_nums):02d}", rc_nums == list(range(1, len(rc_nums) + 1))),
    ("One text_en per reason_group", DRAFT.groupby("reason_group").text_en.nunique().eq(1).all()),
    ("suggested_by in {DICTIONARY, NAME_ONLY}", DRAFT.suggested_by.isin(["DICTIONARY", "NAME_ONLY"]).all()),
    ("suggested_by consistent with match status",
     (DRAFT.suggested_by.eq("DICTIONARY") == DRAFT.feature.map(matched_of)).all()),
    ("DICTIONARY rows have dict_description; NAME_ONLY rows do not",
     DRAFT.loc[DRAFT.suggested_by == "DICTIONARY", "dict_description"].notna().all()
     and DRAFT.loc[DRAFT.suggested_by == "NAME_ONLY", "dict_description"].isna().all()),
    ("text_ar empty", back.text_ar.eq("").all()),
    ("reason_code_map.csv not created/modified",
     final_now == FINAL_EXISTED_BEFORE and final_mtime_now == FINAL_MTIME_BEFORE),
]
try:
    pd.testing.assert_frame_equal(back[OUT_COLS].reset_index(drop=True), exp[OUT_COLS].reset_index(drop=True),
                                  check_dtype=False, check_exact=False, rtol=1e-12)
    _same = True
except AssertionError as e:
    print(f"Re-read mismatch: {e}"); _same = False
checks.append(("Re-read CSV equals in-memory table", _same))

PASS_S, FAIL_S = "\u2705 PASS", "\u274c FAIL"
all_pass = True
for desc, ok in checks:
    ok = bool(ok); all_pass &= ok
    print(f"  {PASS_S if ok else FAIL_S} : {desc}")

summary = {
    "module": "s05_reason_code_map_draft", "spec": "13.2",
    "run_time": datetime.now().isoformat(timespec="seconds"), "runtime_sec": round(time.time() - _t0, 1),
    "git_commit": GIT_COMMIT, "config_hash": CONFIG_HASH,
    "counts": {"pool_entries": len(POOL), "special_indicators_excluded": len(SPECIAL_IND),
               "rows": int(len(DRAFT)), "groups": int(DRAFT.reason_group.nunique()),
               "dictionary_matched": int((DRAFT.suggested_by == "DICTIONARY").sum()),
               "name_only": int((DRAFT.suggested_by == "NAME_ONLY").sum()),
               "in_model_A": int((DRAFT.final_A == "Y").sum())},
    "features_per_group": {r.reason_group: int(r.n_features) for r in GROUP_SUMMARY.itertuples()},
    "unmatched_features": UNMATCHED.feature.tolist(),
    "single_feature_groups": SINGLETONS.reason_group.tolist(),
    "output": DRAFT_CSV, "final_map_written": False,
    "checks": {d: bool(o) for d, o in checks}, "pass_fail": "PASS" if all_pass else "FAIL",
}
_tmp_js = os.path.join(tempfile.mkdtemp(), "s05_reason_code_map_draft.json")
with open(_tmp_js, "w") as fh:
    json.dump(summary, fh, indent=2)
shutil.copyfile(_tmp_js, os.path.join(RUN_SUMS, "s05_reason_code_map_draft.json"))
print(f"\nRun summary: {os.path.join(RUN_SUMS, 's05_reason_code_map_draft.json')}")

failed = [d for d, o in checks if not o]
assert not failed, f"ACCEPTANCE FAILED: {failed}"
print("\n\u2705 All acceptance criteria passed.")